In [44]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used

# Energy emission peaks in MeV
Cs137_MeV = 0.662
Na22_MeV = [0.511, 1.275, 1.786]
Co60_MeV = [1.173, 1.332, 2,505]
Am241_MeV = 0.0595
# Background emission
K40_MeV = 1.460
Tl208_MEV = 2.614

colors = [
    ROOT.kRed + 1,
    ROOT.kBlue + 1,
    ROOT.kGreen + 2,
    ROOT.kOrange + 7,
    ROOT.kMagenta + 1,
    ROOT.kCyan + 1,
    ROOT.kViolet + 2,
    ROOT.kPink + 7,
    ROOT.kTeal + 3,
    ROOT.kAzure + 2,
    ROOT.kSpring + 5,
]
siphras = ['A', 'B']

In [26]:
files = sorted((project_root/'data/260518').glob('SUBT_*'))
acqs = [SiphraAcquisition(file) for file in files]
#print(acqs)
Abackground = acqs[12]
Bbackground = acqs[25]
backgrounds = [acqs[12], acqs[25]]
calibrations = [acqs[0], acqs[13]]
Aacqs = acqs[:12]
Bacqs = acqs[13:25]
acqs = [acqs[1:12], acqs[14:25]]
print(backgrounds)
print(calibrations)
print(acqs)

[SIPHRAAcquisition(File: 'SUBT_A18_CollimatedSource_Background'), SIPHRAAcquisition(File: 'SUBT_B18_CollimatedSource_Background')]
[SIPHRAAcquisition(File: 'SUBT_A06_CollimatedSource_Na22_calibration'), SIPHRAAcquisition(File: 'SUBT_B06_CollimatedSource_Na22_calibration')]
[[SIPHRAAcquisition(File: 'SUBT_A07_CollimatedSource_Cs137_3cm'), SIPHRAAcquisition(File: 'SUBT_A08_CollimatedSource_Cs137_4cm'), SIPHRAAcquisition(File: 'SUBT_A09_CollimatedSource_Cs137_5cm'), SIPHRAAcquisition(File: 'SUBT_A10_CollimatedSource_Cs137_6cm'), SIPHRAAcquisition(File: 'SUBT_A11_CollimatedSource_Cs137_7cm'), SIPHRAAcquisition(File: 'SUBT_A12_CollimatedSource_Cs137_8cm'), SIPHRAAcquisition(File: 'SUBT_A13_CollimatedSource_Cs137_9cm'), SIPHRAAcquisition(File: 'SUBT_A14_CollimatedSource_Cs137_10cm'), SIPHRAAcquisition(File: 'SUBT_A15_CollimatedSource_Cs137_11cm'), SIPHRAAcquisition(File: 'SUBT_A16_CollimatedSource_Cs137_12cm'), SIPHRAAcquisition(File: 'SUBT_A17_CollimatedSource_Cs137_13cm')], [SIPHRAAcquisit

In [50]:
# histograms
hists = {}
sources = []
distances = ['3cm', '4cm', '5cm', '6cm', '7cm', '8cm', '9cm', '10cm', '11cm', '12cm', '13cm']
for i in range(2):
    siphra = siphras[i]
    bg = backgrounds[i]
    hist_bg = ROOT.TH1F(f"{siphra} Background", "", BITS12, 0 , BITS12)
    hist_bg.Fill(bg['s']/len(bg.active_chs))
    hist_bg.GetXaxis().SetTitle("ADC channel number")
    hist_bg.GetYaxis().SetTitle(r"Normalized counts")
    hists[f"{siphra}_BG"] = hist_bg
    #hists = {}
    
    for sgnl, distance in zip(acqs[i], distances):
        filepath = sgnl.filepath.stem
        #src = (MetadataLoader.load(sgnl.metadataFile)).source
        #print(src)
        print(distance)
        sources.append(src)
        # Signal and Background
        hist_sgnl = ROOT.TH1F(f"{distance+siphra} Signal", "", BITS12, 0, BITS12)
        
        hist_sgnl.Fill(sgnl['s']/len(sgnl.active_chs))
        
        hist_sgnl.Scale(bg.exposure/sgnl.exposure)
        # Clean spectrum
        hist_clean = hist_sgnl.Clone(f"{distance+siphra} Clean")
        hist_clean.Add(hist_bg, -1)
        for hist in [hist_sgnl, hist_clean]:
            #hist.SetTitle(r"^{22}Na spectra - CI gain = 1/0.75 pC")
            hist.GetXaxis().SetTitle("ADC channel number")
            hist.GetYaxis().SetTitle(r"Normalized counts")
        hists[distance+siphra] = hist_sgnl
        
        hists[f"{distance+siphra}_clean"] = hist_clean
        del hist_sgnl
    del hist_bg

3cm
4cm
5cm
6cm
7cm
8cm
9cm
10cm
11cm
12cm
13cm
3cm
4cm
5cm
6cm
7cm
8cm
9cm
10cm
11cm
12cm
13cm


In [53]:
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()

Yinit = 0.82 # For stat boxes

cv1 = ROOT.TCanvas('cv1', 'cv1', 2400, 1200)
cv1.Divide(3, 4)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(11)]

for i, siphra in enumerate(siphras):
    
    

    for idx, (lg, distance) in enumerate(zip(lgds, distances)):   
        cv.cd(idx+1)
        
        
        sgnl = hists[distance+siphra] 
        bg = hists[f"{siphra}_BG"] 
        clean = hists[f"{distance+siphra}_clean"]
        
        lg.AddEntry(sgnl, "Signal")
        lg.AddEntry(bg, "Background")
        lg.AddEntry(clean, "Bg subtracted " + distance + " SIPHRA "+siphra)
        sgnl.SetLineColor(colors[0])
        bg.SetLineColor(colors[1])
        clean.SetLineColor(colors[2])
        sgnl.SetTitle(src)
        bg.Draw("hist")
        
        sgnl.Draw("sames hist")
        clean.Draw("sames hist")
        ROOT.gPad.Update()
        """for i, sp in enumerate([sgnl, bg, clean]):
            st = sp.FindObject("stats")
            # print(type(st))
            st.SetY1NDC(Yinit - i*0.08)
            st.SetY2NDC(0.1)"""
    lg.Draw()
    cv1.cd(idx+1).SetLogy()
    cv1.Draw()

In [45]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

Yinit = 0.82 # For stat boxes

cv = ROOT.TCanvas('cv', 'cv', 2400, 1200)
cv.Divide(2,1)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(2)]

for i, (siphra, lg) in enumerate(zip(siphras, lgds)):
    cv.cd(i+1)
    

    for idx, (distance) in enumerate(distances):   
        
        
        #sgnl = hists[f"{distance+siphra}_clean"]
        #bg = hists[src+'_BG']
        clean = hists[f"{distance+siphra}_clean"]
        
        #lg.AddEntry(sgnl, "Signal")
        #lg.AddEntry(bg, "Background")
        lg.AddEntry(clean, "Bg subtracted " + distance + " SIPHRA "+siphra)
        #sgnl.SetLineColor(colors[0])
        #bg.SetLineColor(colors[1])
        clean.SetLineColor(colors[idx])
        #sgnl.SetTitle(src)
        if idx ==0:
            clean.Draw("hist")
        else:
            
            #bg.Draw("sames hist")
            clean.Draw("sames hist")
        ROOT.gPad.Update()
        """for i, sp in enumerate([sgnl, bg, clean]):
            st = sp.FindObject("stats")
            # print(type(st))
            st.SetY1NDC(Yinit - i*0.08)
            st.SetY2NDC(0.1)"""
    lg.Draw()
    cv.cd(i+1).SetLogy()
    cv.Draw()

In [42]:
from analysis import *

#Calibration based on the 3 peaks of the Na-22 spectrum


hist = hists['Cs-137_clean']
energy_ranges = [(0,1), (100, 250)]
energies = [0, Cs137_MeV]

c_fit = calibration_fit(hist, energy_ranges, energies)
print(c_fit)

KeyError: 'Cs-137_clean'

In [6]:
#Calibrating Cs137 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[2], BITS12)
hist_cal_bg_Cs137.Scale(1/acqs[2].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[1], BITS12)
hist_cal_src_Cs137.Scale(1/acqs[1].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)


# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1600, 800)
cv1.Divide(2,1)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv1.cd(1)
hist_cal_bg_Cs137.SetLineColor(colors[0])
hist_cal_src_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"Signal ^{137}Cs", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
lg1.Draw()
cv1.cd(1).SetLogy()
#hist_cal_bg.GetYaxis().SetRangeUser(0,1e4)


lg2 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv1.cd(2)
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
hist_cal_clean_Cs137.Draw("hist")
lg2.AddEntry(hist_cal_clean_Cs137, r"^{137}Cs bg subtracted", "l")
lg2.Draw()
cv1.cd(2).SetLogy()

cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, [(0.3, 0.9)], [Cs137_MeV])
print(res_Cs137)



[0.3761890893184299]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      518.581
NDf                       =          164
Edm                       =  1.52714e-08
NCalls                    =           73
Constant                  =      25.7054   +/-   0.0782673   
Mean                      =      0.66185   +/-   0.000273606 
Sigma                     =     0.105748   +/-   0.000220684  	 (limited)


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [7]:
#Calibration based on the 3 peaks of the Na-22 spectrum
hist = hists['Na22_clean']
energy_ranges = [(150, 200), (320, 420), (450, 520)]
energies = Na22_MeV

c_fit = calibration_fit(hist, energy_ranges, energies)

#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Na22 = calibrated_histogram(c_fit, acqs[4], BITS12)
hist_cal_bg_Na22.Scale(1/acqs[4].exposure) 
hist_cal_src_Na22 = calibrated_histogram(c_fit, acqs[3], BITS12)
hist_cal_src_Na22.Scale(1/acqs[3].exposure) 
hist_cal_clean_Na22 = hist_cal_src_Na22.Clone("Calibrated signal no BG")
hist_cal_clean_Na22.Add(hist_cal_bg_Na22, -1)


# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv2'):
    ROOT.gROOT.FindObject('cv2').Close()
cv2 = ROOT.TCanvas("cv2", "cv2", 1600, 800)
cv2.Divide(2,1)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv2.cd(1)
hist_cal_bg_Na22.SetLineColor(colors[0])
hist_cal_src_Na22.SetLineColor(colors[1])
hist_cal_src_Na22.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Na22.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Na22, "Background", "l")
lg1.AddEntry(hist_cal_src_Na22, r"Signal ^{22}Na", "l")
hist_cal_src_Na22.Draw("hist")
hist_cal_bg_Na22.Draw("sames hist")
lg1.Draw()
cv2.cd(1).SetLogy()
#hist_cal_bg.GetYaxis().SetRangeUser(0,1e4)


lg2 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv2.cd(2)
hist_cal_clean_Na22.SetLineColor(colors[2])
hist_cal_clean_Na22.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_clean_Na22.GetYaxis().SetTitle(r"Count rate [Hz]")
hist_cal_clean_Na22.Draw("hist")
lg2.AddEntry(hist_cal_clean_Na22, r"^{22}Na bg subtracted", "l")
lg2.Draw()
cv2.cd(2).SetLogy()

cv2.Draw()

#Calculate energy resolution
res_Na22 = energy_resolution(hist_cal_clean_Na22, [(0.13, 0.6), (1.1, 1.5), (1.5 , 2.1)],  Na22_MeV)
print(res_Na22)



[0.6098052497160661, 0.3310662458402524, 0.19985352745576823]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =       50.431
NDf                       =           47
Edm                       =  6.26293e-06
NCalls                    =          110
Constant                  =      187.956   +/-   4.04943     
Mean                      =      177.384   +/-   1.67095     
Sigma                     =      37.5292   +/-   4.35406      	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      152.171
NDf                       =           97
Edm                       =  2.80563e-08
NCalls                    =           96
Constant                  =      606.675   +/-   4.80363     
Mean                      =       371.93   +/-   0.452134    
Sigma                     =      45.3973   +/-   0.782852     	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
C

Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [8]:
#Calibration based on the 3 peaks of the Na-22 spectrum
hist = hists['Am-241_clean']
energy_ranges = [(0, 1), (5, 30)]
energies = [0, Am241_MeV]

c_fit = calibration_fit(hist, energy_ranges, energies)
#print(c_fit)

#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Am241  = calibrated_histogram(c_fit, acqs[8], BITS12)
hist_cal_bg_Am241.Scale(1/acqs[8].exposure) 
hist_cal_src_Am241 = calibrated_histogram(c_fit, acqs[7], BITS12)
hist_cal_src_Am241.Scale(1/acqs[7].exposure) 
hist_cal_clean_Am241 = hist_cal_src_Am241.Clone("Calibrated signal no BG")
hist_cal_clean_Am241.Add(hist_cal_bg_Am241, -1)


# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv4'):
    ROOT.gROOT.FindObject('cv4').Close()
cv4 = ROOT.TCanvas("cv4", "cv4", 1600, 800)
cv4.Divide(2,1)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv4.cd(1)
hist_cal_bg_Am241.SetLineColor(colors[0])
hist_cal_src_Am241.SetLineColor(colors[1])
lg1.AddEntry(hist_cal_bg_Am241, "Background", "l")
lg1.AddEntry(hist_cal_src_Am241, r"Signal ^{241}Am", "l")
hist_cal_src_Am241.Draw("hist")
hist_cal_bg_Am241.Draw("sames hist")
lg1.Draw()
cv4.cd(1).SetLogy()
#hist_cal_bg.GetYaxis().SetRangeUser(0,1e4)


lg2 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv4.cd(2)
hist_cal_clean_Am241.SetLineColor(colors[2])
hist_cal_clean_Am241.Draw("hist")
lg2.AddEntry(hist_cal_clean_Am241, r"^{241}Am bg subtracted", "l")
lg2.Draw()
cv4.cd(2).SetLogy()

cv4.Draw()

#Calculate energy resolution
res_Am241 = energy_resolution(hist_cal_clean_Am241, [(0.01, 0.09)],  [Am241_MeV])
print(res_Am241)



[0.5258062473970452]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =  8.89298e-09
NDf                       =            0
Edm                       =  8.89072e-09
NCalls                    =           57
Constant                  =      3.63942   +/-   76.2026     
Mean                      =     0.316732   +/-   5.06048     
Sigma                     =     0.251842   +/-   1.78535      	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      26.1098
NDf                       =           22
Edm                       =  1.46194e-05
NCalls                    =           71
Constant                  =      491.938   +/-   33.4685     
Mean                      =      20.7681   +/-   0.42253     
Sigma                     =      4.70087   +/-   0.220973     	 (limited)
****************************************
Minimizer is Linear / Migrad
Chi2                      =   6.9514e-35
ND

Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
